# 01 — Data Cleaning

Game-On is a semantic search engine: every game gets turned into a text blob and embedded with SBERT, so search quality depends entirely on how clean and information-dense that text is. This notebook explores the raw Steam dataset's problems and fixes them.

Pipeline: **01 cleaning → 02 feature engineering → 03 EDA → 04 modeling**.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re

## 2. Load the raw data

In [2]:
CSV_PATH = '../raw_data/steam_games.csv'
CSV_PATH_IMG = '../raw_data/applications.csv'

df = pd.read_csv(CSV_PATH)
df1 = pd.read_csv(CSV_PATH_IMG, low_memory=False)

df.shape

(40833, 20)

## 3. Exploring what's wrong (before cleaning)

The columns that matter most here are the ones that will end up in the embedding text: `genre`, `popular_tags`, `game_details`, `game_description`. If those are messy or missing, the embeddings — and therefore the search results — suffer.

In [3]:
# How much of the text that will feed the embeddings is missing?
df[['genre', 'popular_tags', 'game_details', 'game_description']].isna().sum()

genre                438
popular_tags        2945
game_details         520
game_description    2913
dtype: int64

In [4]:
# Raw description: markdown-style asterisks and inconsistent whitespace
df['game_description'].iloc[0]

' About This Game Developed by id software, the studio that pioneered the first-person shooter genre and created multiplayer Deathmatch, DOOM returns as a brutally fun and challenging modern-day shooter experience. Relentless demons, impossibly destructive guns, and fast, fluid movement provide the foundation for intense, first-person combat – whether you’re obliterating demon hordes through the depths of Hell in the single-player campaign, or competing against your friends in numerous multiplayer modes. Expand your gameplay experience using DOOM SnapMap game editor to easily create, play, and share your content with the world. STORY: You’ve come here for a reason. The Union Aerospace Corporation’s massive research facility on Mars is overwhelmed by fierce and powerful demons, and only one person stands between their world and ours.  As the lone DOOM Marine, you’ve been activated to do one thing – kill them all. KEY FEATURES: A Relentless Campaign There is no taking cover or stopping t

In [5]:
# Price and release_date come in as free-form strings, not numbers/dates
df[['original_price', 'release_date']].drop_duplicates().sample(10, random_state=1)

,original_price,release_date
18402,$59.99,"Nov 30, 2012"
20340,$2.99,"Aug 29, 2016"
25241,Free to Play,"Jan 30, 2017"
29579,$59.99,"Jun 20, 2018"
32932,NaN,About a year
11801,3DCoat Modding Tool Demo,"Feb 27, 2018"
15581,$14.99,"Jul 11, 2016"
4127,$9.99,"Dec 3, 2007"
30878,$6.99,"Mar 10, 2019"
2460,$14.99,"Oct 25, 2010"


In [6]:
# Review data is buried inside a sentence instead of being its own column
df['all_reviews'].dropna().iloc[0]

'Very Positive,(42,550),- 92% of the 42,550 user reviews for this game are positive.'

In [7]:
# Any exact duplicate games?
df.duplicated(subset=['name', 'url']).sum()

np.int64(0)

## 4. Cleaning, step by step

**Reviews** — extract the review count and percentage out of the free-text `all_reviews` column

In [24]:
df['total_review'] = df['all_reviews'].str.extract(r'\(([\d,]+)\)').iloc[:, 0].str.replace(',', '')
df['total_review'] = pd.to_numeric(df['total_review'], errors='coerce')
df['review_per'] = df['all_reviews'].str.extract(r'(\d+)%').iloc[:, 0]
df['review_per'] = pd.to_numeric(df['review_per'], errors='coerce')

**Drop columns** that don't add value for search or recommendation

In [25]:
df = df.drop(columns=['types', 'all_reviews', 'desc_snippet', 'recent_reviews', 'developer',
                       'publisher', 'achievements', 'mature_content', 'minimum_requirements',
                       'recommended_requirements', 'discount_price'])

**Comma-separated columns** — add a space after each comma so the text reads naturally once it's fed into the embedding

In [26]:
for col in ['genre', 'popular_tags', 'game_details', 'languages']:
    df[col] = df[col].str.replace(',', ', ', regex=False)

**Release date** — drop rows without a real year, then normalize whatever's left down to a single 4-digit year

In [27]:
df = df[df['release_date'].str.contains(r'\d{4}', na=False)]
df['release_date'] = df['release_date'].str.replace(r'(\d{4})\d{4}', r'\1', regex=True).str.findall(r'\d{4}').str[-1]
df['release_date'] = df['release_date'].str.extract(r'(\d{4})(?!.*\d{4})')
df['release_date'] = df['release_date'].astype(int)

**App ID** — extracted from the Steam store URL, needed to merge with the second dataset and to call the Steam API later

In [28]:
df['appid'] = df['url'].str.extract(r'/app/(\d+)')

**Description** — strip markdown asterisks and collapse whitespace (this is exactly the noise we saw in section 3)

In [29]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = re.sub(r"\*+|\s+", ' ', text)
    text = text.lower()
    return text.strip()

df['game_description'] = df['game_description'].apply(clean_text)
df

,url,name,release_date,popular_tags,game_details,languages,genre,game_description,original_price,total_review,review_per,appid
0,https://store.steampowered.com/app/379720/DOOM/,DOOM,2016,"FPS, Gore, Action, Demons, Shooter, First-Pers...","Single-player, Multi-player, Co-op, Steam Achi...","English, French, Italian, German, Spanish - Sp...",Action,"about this game developed by id software, the ...",$19.99,42550.0,92.0,379720
1,https://store.steampowered.com/app/578080/PLAY...,PLAYERUNKNOWN'S BATTLEGROUNDS,2017,"Survival, Shooter, Multiplayer, Battle Royale,...","Multi-player, Online Multi-Player, Stats","English, Korean, Simplified Chinese, French, G...","Action, Adventure, Massively Multiplayer",about this game playerunknown's battlegrounds ...,$29.99,836608.0,49.0,578080
2,https://store.steampowered.com/app/637090/BATT...,BATTLETECH,2018,"Mechs, Strategy, Turn-Based, Turn-Based Tactic...","Single-player, Multi-player, Online Multi-Play...","English, French, German, Russian","Action, Adventure, Strategy",about this game from original battletech/mechw...,$39.99,7030.0,71.0,637090
3,https://store.steampowered.com/app/221100/DayZ/,DayZ,2018,"Survival, Zombies, Open World, Multiplayer, Pv...","Multi-player, Online Multi-Player, Steam Works...","English, French, Italian, German, Spanish - Sp...","Action, Adventure, Massively Multiplayer",about this game the post-soviet country of che...,$44.99,167115.0,61.0,221100
4,https://store.steampowered.com/app/8500/EVE_On...,EVE Online,2003,"Space, Massively Multiplayer, Sci-fi, Sandbox,...","Multi-player, Online Multi-Player, MMO, Co-op,...","English, German, Russian, French","Action, Free to Play, Massively Multiplayer, R...",about this game,Free,11481.0,74.0,8500
...,...,...,...,...,...,...,...,...,...,...,...,...
40828,https://store.steampowered.com/app/899836/Rock...,Rocksmith® 2014 Edition – Remastered – Sabaton...,2019,"Casual, Simulation","Single-player, Shared/Split Screen, Downloadab...","English, German, French, Italian, Spanish - Sp...","Casual, Simulation","about this content play ""ghost division"" by sa...",$2.99,NaN,NaN,899836
40829,https://store.steampowered.com/app/899832/Rock...,Rocksmith® 2014 Edition – Remastered – Stone T...,2019,"Casual, Simulation","Single-player, Shared/Split Screen, Downloadab...","English, German, French, Italian, Spanish - Sp...","Casual, Simulation","about this content play ""trippin’ on a hole in...",$2.99,NaN,NaN,899832
40830,https://store.steampowered.com/app/906840/Fant...,Fantasy Grounds - Quests of Doom 4: A Midnight...,2018,"RPG, Indie, Strategy, Software, Turn-Based, Fa...","Multi-player, Co-op, Cross-Platform Multiplaye...",English,"Indie, RPG, Strategy",about this content quests of doom 4: a midnigh...,$7.99,NaN,NaN,906840
40831,https://store.steampowered.com/app/906635/Mega...,Mega Man X5 Sound Collection,2018,Action,"Single-player, Downloadable Content, Steam Ach...","English, French, Italian, German, Spanish - Sp...",Action,about this content get equipped with the stunn...,$9.99,NaN,NaN,906635


**Price** — strip the `$`, coerce to numeric, treat `Free` as `0`

In [30]:
df['original_price'] = df['original_price'].replace({'Free': '0', '0': '0'})
df['original_price'] = df['original_price'].astype(str).str.replace('$', '', regex=False)
df['original_price'] = pd.to_numeric(df['original_price'], errors='coerce')
df['original_price'] = df['original_price'].fillna(0.0)

### Merge with the app-details dataset

The second dataset adds `header_image`, `required_age` and `metacritic_score` — all keyed by `appid`.

In [31]:
df1 = df1[['header_image', 'required_age', 'short_description', 'appid', 'metacritic_score']].copy()

# '17+' -> 17, invalid values (e.g. leftover JS from scraping) -> 0
df1['required_age'] = df1['required_age'].astype(str).str.replace('+', '', regex=False)
df1['required_age'] = pd.to_numeric(df1['required_age'], errors='coerce').fillna(0).astype(int)
df1['required_age'] = df1['required_age'].apply(
    lambda x: "For all ages" if x == 0 else f"Game for people over {x} years old"
)

df['appid'] = df['appid'].astype(str)
df1['appid'] = df1['appid'].astype(str)

df = pd.merge(df, df1, on='appid', how='inner')
df.head(3)

,url,name,release_date,popular_tags,game_details,languages,genre,game_description,original_price,total_review,review_per,appid,header_image,required_age,short_description,metacritic_score
0,https://store.steampowered.com/app/379720/DOOM/,DOOM,2016,"FPS, Gore, Action, Demons, Shooter, First-Pers...","Single-player, Multi-player, Co-op, Steam Achi...","English, French, Italian, German, Spanish - Sp...",Action,"about this game developed by id software, the ...",19.99,42550.0,92.0,379720,https://shared.akamai.steamstatic.com/store_it...,Game for people over 17 years old,Now includes all three premium DLC packs (Unto...,85.0
1,https://store.steampowered.com/app/578080/PLAY...,PLAYERUNKNOWN'S BATTLEGROUNDS,2017,"Survival, Shooter, Multiplayer, Battle Royale,...","Multi-player, Online Multi-Player, Stats","English, Korean, Simplified Chinese, French, G...","Action, Adventure, Massively Multiplayer",about this game playerunknown's battlegrounds ...,29.99,836608.0,49.0,578080,https://shared.akamai.steamstatic.com/store_it...,For all ages,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",NaN
2,https://store.steampowered.com/app/637090/BATT...,BATTLETECH,2018,"Mechs, Strategy, Turn-Based, Turn-Based Tactic...","Single-player, Multi-player, Online Multi-Play...","English, French, German, Russian","Action, Adventure, Strategy",about this game from original battletech/mechw...,39.99,7030.0,71.0,637090,https://shared.akamai.steamstatic.com/store_it...,For all ages,Take command of your own mercenary outfit of '...,78.0


## 5. Save the cleaned dataset

`02_feature_engineering.ipynb` picks up from here to build the embedding text and `quality_score`.

In [32]:
df.to_pickle('../game_on/data/df_clean_step1.pkl')
print(f"Saved {len(df)} cleaned games")

Saved 32313 cleaned games
